In [3]:
# ==============================================================================
# 1. IMPORTS
# ==============================================================================

# OpenPIV
from openpiv import windef
from openpiv import tools, scaling, validation, filters, preprocess
import openpiv.pyprocess as process
from openpiv import pyprocess

# General
import numpy as np
import tifffile as tif
import matplotlib.pyplot as plt
import pandas as pd
from roifile import ImagejRoi

import pathlib
from time import time
import warnings
%matplotlib qt

# Custom functions
from src.PIV import run_PIV_on_frames

In [ ]:
# ==============================================================================
# 4. REFERENCE FRAME COMPARISON
# ==============================================================================

# SECTION Standard unstressed reference frame 

# Load data 
data_path = '/mnt/crunch/Clark/Fly_TFM/data/first/reference-left_the_frame/' 
filename = 'PIVlab_0350.txt' 

data = pd.read_csv(data_path + filename, skiprows=2)

# Get column values 
x_piv = data['x [px]'].values
y_piv = data['y [px]'].values
u_piv = data['u [px/frame]'].values
v_piv = data['v [px/frame]'].values

print(x_piv.shape, y_piv.shape, u_piv.shape, v_piv.shape)

# Load manual centerline for now
roi = ImagejRoi.fromfile('/mnt/crunch/Clark/Fly_TFM/data/first/0001-1216-0912.roi')
centerline_points = roi.coordinates() # numpy array of (x, y)

# Visualize 
plt.figure(figsize=(10, 10))
plt.quiver(x_piv, y_piv, u_piv, v_piv)
plt.plot(centerline_points[:, 0], centerline_points[:, 1], 'r-o')
plt.axis('equal')
points = plt.ginput(n=-1, timeout=0) 
plt.close()

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/crunch/Clark/Fly_TFM/data/first/reference=left_the_frame/PIVlab_0350.txt'

In [ ]:
from matplotlib.path import Path as MplPath

polygon = MplPath(points)
inside = polygon.contains_points(np.column_stack([x_piv, y_piv]))

near_larva_mask = inside
far_field_mask = ~inside

# Visualize gridpoints that were kept vs excluded from my terrible clicking
plt.figure(figsize=(10, 10))
plt.scatter(x_piv[inside], y_piv[inside], c='red', s=10, label='near (excluded)')
plt.scatter(x_piv[~inside], y_piv[~inside], c='blue', s=10, label='far (kept)')
plt.axis('equal')
plt.legend()
plt.show()

# Visualize only the backgroun signal vectors (far field)
plt.figure(figsize=(10,10))
plt.quiver(x_piv[~inside], y_piv[~inside], u_piv[~inside], v_piv[~inside])
plt.axis('equal')
plt.show()

In [ ]:
# Fit the data

def design_matrix(x, y):
    return np.column_stack([np.ones_like(x), x, y, x**2, y**2, x*y])

A_far = design_matrix(x_piv[~inside], y_piv[~inside])

coeffs_u, *_ = np.linalg.lstsq(A_far, u_piv[~inside], rcond=None)
coeffs_v, *_ = np.linalg.lstsq(A_far, v_piv[~inside], rcond=None)

# evaluate the fit everywhere, including inside the excluded blob
A_all = design_matrix(x_piv, y_piv)
u_fit = A_all @ coeffs_u
v_fit = A_all @ coeffs_v

In [ ]:
# Evaluate goodness of fit

u_fit_inside = u_fit[inside]
u_actual_inside = u_piv[inside]

v_fit_inside = v_fit[inside]
v_actual_inside = v_piv[inside]

same_sign_u = np.sign(u_fit_inside) == np.sign(u_actual_inside)
same_sign_v = np.sign(v_fit_inside) == np.sign(v_actual_inside)

print(f"u: {same_sign_u.mean()*100:.0f}% of near-larva points share sign with the fit")
print(f"v: {same_sign_v.mean()*100:.0f}% of near-larva points share sign with the fit")

u: 43% of near-larva points share sign with the fit
v: 45% of near-larva points share sign with the fit


In [ ]:
# Visualize the 3d surface data (for u, x-component)

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Actual far-field data points
ax.scatter(x_piv[~inside], y_piv[~inside], u_piv[~inside], c='blue', s=5, label='actual (far-field)')

# Fitted surface, evaluated on a smooth grid for a clean mesh
xx, yy = np.meshgrid(
    np.linspace(x_piv.min(), x_piv.max(), 50),
    np.linspace(y_piv.min(), y_piv.max(), 50)
)
zz = (coeffs_u[0] + coeffs_u[1]*xx + coeffs_u[2]*yy 
    + coeffs_u[3]*xx**2 + coeffs_u[4]*yy**2 + coeffs_u[5]*xx*yy)

ax.plot_surface(xx, yy, zz, alpha=0.3, color='orange')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('u')
plt.show()

In [ ]:
# Visualize the 3d surface data (for v, y-component)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(x_piv[~inside], y_piv[~inside], v_piv[~inside], c='blue', s=5, label='actual (far-field)')

zz_v = (coeffs_v[0] + coeffs_v[1]*xx + coeffs_v[2]*yy 
        + coeffs_v[3]*xx**2 + coeffs_v[4]*yy**2 + coeffs_v[5]*xx*yy)

ax.plot_surface(xx, yy, zz_v, alpha=0.3, color='orange')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('v')
plt.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=x_piv[~inside], y=y_piv[~inside], z=u_piv[~inside],
    mode='markers', marker=dict(size=2, color='blue'), name='actual (far-field)'
))

fig.add_trace(go.Surface(
    x=xx, y=yy, z=zz, opacity=0.4, colorscale='Oranges', showscale=False, name='fit'
))

fig.update_layout(scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='u'))
fig.write_html('u_surface_check.html')

In [ ]:
fig_v = go.Figure()

fig_v.add_trace(go.Scatter3d(
    x=x_piv[~inside], y=y_piv[~inside], z=v_piv[~inside],
    mode='markers', marker=dict(size=2, color='blue'), name='actual (far-field)'
))

fig_v.add_trace(go.Surface(
    x=xx, y=yy, z=zz_v, opacity=0.4, colorscale='Oranges', showscale=False, name='fit'
))

fig_v.update_layout(scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='v'))
fig_v.write_html('v_surface_check.html')

In [ ]:
u_residual = u_piv - u_fit
v_residual = v_piv - v_fit

# Visualize only the backgroun signal vectors (far field)
plt.figure(figsize=(10,10))
plt.quiver(x_piv[~inside], y_piv[~inside], u_piv[~inside], v_piv[~inside])
plt.axis('equal')
plt.show()

plt.figure(figsize=(10,10))
plt.quiver(x_piv[~inside], y_piv[~inside], u_residual[~inside], v_residual[~inside])
plt.axis('equal')
plt.show()


# Print average vector magnitude in far field and near field region of actual data
print(f"Average vector magnitude in far field (actual): {np.sqrt(u_piv[~inside]**2 + v_piv[~inside]**2).mean():.4f}")
print(f"Average vector magnitude in near field (actual): {np.sqrt(u_piv[inside]**2 + v_piv[inside]**2).mean():.4f}")

# Print average vector magnitude in far field and near field region of residual
print(f"Average vector magnitude in far field (residual): {np.sqrt(u_residual[~inside]**2 + v_residual[~inside]**2).mean():.4f}")
print(f"Average vector magnitude in near field (residual): {np.sqrt(u_residual[inside]**2 + v_residual[inside]**2).mean():.4f}")

Average vector magnitude in far field (actual): 2.6441
Average vector magnitude in near field (actual): 4.0594
Average vector magnitude in far field (residual): 1.0311
Average vector magnitude in near field (residual): 4.0150


In [ ]:
u_residual2 = u_residual.copy()
v_residual2 = v_residual.copy()

u_residual2[inside] = u_residual[inside] - u_piv[inside]
v_residual2[inside] = v_residual[inside] - v_piv[inside]

plt.figure(figsize=(10,10))
plt.quiver(x_piv, y_piv, u_residual2, v_residual2)
plt.axis('equal')
plt.show()

In [5]:
# Now just compare subtractions

# for current_frame in range(350, 381):
#     vid_1_frame = current_frame
#     vid_2_frame = current_frame - 349

#     vid_1_data_path = '/mnt/crunch/Clark/Fly_TFM/data/first/reference-left_the_frame/' + f'PIVlab_{vid_1_frame:04d}.txt'
#     vid_2_data_path = '/mnt/crunch/Clark/Fly_TFM/data/first/reference-350/' + f'PIVlab_{vid_2_frame:04d}.txt'
    
#     print(f"vid_1_frame: {vid_1_frame}, vid_2_frame: {vid_2_frame}")
#     # print(f"vid_1_data_path: {vid_1_data_path}")
#     # print(f"vid_2_data_path: {vid_2_data_path}")
    
#     # Load 
#     vid_1_data = pd.read_csv(vid_1_data_path, skiprows=2)
#     vid_2_data = pd.read_csv(vid_2_data_path, skiprows=2)
    
#     # Get column values
#     x_piv_1 = vid_1_data['x [px]'].values
#     y_piv_1 = vid_1_data['y [px]'].values
#     u_piv_1 = vid_1_data['u [px/frame]'].values
#     v_piv_1 = vid_1_data['v [px/frame]'].values
    
#     x_piv_2 = vid_2_data['x [px]'].values
#     y_piv_2 = vid_2_data['y [px]'].values
#     u_piv_2 = vid_2_data['u [px/frame]'].values
#     v_piv_2 = vid_2_data['v [px/frame]'].values
    
#     # print(np.allclose(x_piv_1, x_piv_2), np.allclose(y_piv_1, y_piv_2))
#     u_diff = u_piv_2 - u_piv_1
#     v_diff = v_piv_2 - v_piv_1

# ------------------------------------------------------------------------------

vid_1_reference_data = pd.read_csv('/mnt/crunch/Clark/Fly_TFM/data/first/reference-left_the_frame/PIVlab_0350.txt', skiprows=2)
x_piv_1 = vid_1_reference_data['x [px]'].values
y_piv_1 = vid_1_reference_data['y [px]'].values
u_piv_1 = vid_1_reference_data['u [px/frame]'].values
v_piv_1 = vid_1_reference_data['v [px/frame]'].values

results = {}  # frame -> (u_diff, v_diff)

for current_frame in range(350, 381):
    vid_1_path = f'/mnt/crunch/Clark/Fly_TFM/data/first/reference-left_the_frame/PIVlab_{current_frame:04d}.txt'
    vid_1_data = pd.read_csv(vid_1_path, skiprows=2)
    
    u_piv_2 = vid_1_data['u [px/frame]'].values
    v_piv_2 = vid_1_data['v [px/frame]'].values
    
    u_diff = u_piv_2 - u_piv_1
    v_diff = v_piv_2 - v_piv_1
    
    results[current_frame] = (u_diff, v_diff)

In [ ]:
import napari

n_frames = 31
n_points = len(x_piv_1)
vectors = np.zeros((n_frames, n_points, 2, 3))
for i, current_frame in enumerate(range(350, 381)):
    u_d, v_d = results[current_frame]
    vectors[i, :, 0, 0] = i
    vectors[i, :, 0, 1] = y_piv_1
    vectors[i, :, 0, 2] = x_piv_1
    vectors[i, :, 1, 0] = 0
    vectors[i, :, 1, 1] = v_d
    vectors[i, :, 1, 2] = u_d
vectors_flat = vectors.reshape(-1, 2, 3)

x_unique = np.unique(x_piv_1)
y_unique = np.unique(y_piv_1)
nx, ny = len(x_unique), len(y_unique)
mag_stack = np.zeros((n_frames, ny, nx))
for i, current_frame in enumerate(range(350, 381)):
    u_d, v_d = results[current_frame]
    mag = np.sqrt(u_d**2 + v_d**2)
    col_idx = np.searchsorted(x_unique, x_piv_1)
    row_idx = np.searchsorted(y_unique, y_piv_1)
    mag_stack[i, row_idx, col_idx] = mag

viewer = napari.Viewer() 
viewer.add_image(mag_stack, name='magnitude', colormap='jet', opacity=0.7)
viewer.add_vectors(vectors_flat, edge_width=1, name='u_v_diff')
napari.run()

In [ ]:
import napari
import numpy as np
from tifffile import imwrite

n_frames = 31  # 350 to 380
n_points = len(x_piv_1)
vectors = np.zeros((n_frames, n_points, 2, 3))

for i, current_frame in enumerate(range(350, 381)):
    u_d, v_d = results[current_frame]
    vectors[i, :, 0, 0] = i 
    vectors[i, :, 0, 1] = y_piv_1 
    vectors[i, :, 0, 2] = x_piv_1
    vectors[i, :, 1, 0] = 0 
    vectors[i, :, 1, 1] = v_d  
    vectors[i, :, 1, 2] = u_d     

vectors_flat = vectors.reshape(-1, 2, 3)


x_unique = np.unique(x_piv_1)
y_unique = np.unique(y_piv_1)
nx, ny = len(x_unique), len(y_unique)
mag_stack = np.zeros((n_frames, ny, nx))

for i, current_frame in enumerate(range(350, 381)):
    u_d, v_d = results[current_frame]
    mag = np.sqrt(u_d**2 + v_d**2)
    col_idx = np.searchsorted(x_unique, x_piv_1)
    row_idx = np.searchsorted(y_unique, y_piv_1)
    mag_stack[i, row_idx, col_idx] = mag

grid_spacing = x_unique[1] - x_unique[0]

print("min:", mag_stack.min(), "max:", mag_stack.max(), "99th pct:", np.percentile(mag_stack, 99))

viewer = napari.Viewer()
viewer.add_image(
    mag_stack, name='magnitude', colormap='jet', opacity=0.7,
    scale=(1, grid_spacing, grid_spacing),
    translate=(0, y_unique[0], x_unique[0]),
    contrast_limits=(0, 5),  
)
viewer.add_vectors(vectors_flat, edge_width=1, name='u_v_diff', length=20)

napari.run()

min: 0.0 max: 32.480660774175334 99th pct: 3.7848839423782414


In [ ]:
screenshots = []
for i in range(n_frames):
    viewer.dims.set_point(0, i)
    screenshot = viewer.screenshot(canvas_only=True)
    screenshots.append(screenshot)

screenshots = np.stack(screenshots)
imwrite('vectors_heatmap_stack.tif', screenshots)